In [ ]:
import pandas as pd
import os
import re
import string
import spacy
from nltk.corpus import stopwords

from transformers import pipeline, logging
# hide all non-critical warnings
logging.set_verbosity_error()

from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Access your API key
api_key = os.getenv("HF_TOKEN")

# print("Your API Key:", api_key)

HUGGING FACE WORKFLOW:

1. Encoder-Only:
- Sentiment Analysis
- Named Entity Recognition
2. Decoder-Only
- VLANText Generation
3. Encoder-Decoder
- Zero-Shot Classification
- Text Summarization

# 1. Sentiment Analysis:

In [4]:
%%timeit
sentiment_analyzer = pipeline(task = "sentiment-analysis",
                              model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
                              device=-1,
                              truncation = True)

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

c:\Users\abhis\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\abhis\.cache\huggingface\hub\models--distilbert--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

1.61 s ± 96.5 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [5]:
#%%timeit
# create full path
file_path = os.path.join(os.getcwd(),'all-data.csv')
pd.set_option('display.max_colwidth',None)
df = pd.read_csv(file_path,encoding="ISO-8859-1")
df.columns = ['lable','Text']
#print(df.head())

In [6]:
# %%timeit
# load spaCy English model
nlp = spacy.load("en_core_web_sm")

# text cleaning function using spaCy
def clean_text_spacy(text):
    # lowercase
    text = text.lower()
    # remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))
    # remove digits
    text = re.sub(r"\d+", "", text)
    # remove extra whitespace
    text = " ".join(text.split())
    # remove stopwords with spaCy
    doc = nlp(text)
    text = " ".join([token.lemma_ for token in doc if not token.is_stop])
    return text

# apply cleaning
df["spacy_cln_txt"] = df["Text"].apply(clean_text_spacy)
df.columns

Index(['lable', 'Text', 'spacy_cln_txt'], dtype='object')

#### SPEEDING UP TRANSFORMERS CODE

- For new Macs with an Apple Silicon chip (i.e., M3 chip), choose device='mps' to use the GPU
- For PCs with a dedicated GPU (i.e., gaming PCs), choose device='cuda' to use your GPU
- For older Macs and most PCs, you’ll only be able to use your CPU for computations (default)

In [7]:
%%time

from transformers import pipeline

sentiment_analyzer = pipeline(
    task = "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english", # 1. smaller model
    device=-1, # running on CPU
    truncation=True,
    use_fast=True # 2. faster tokenization
)

import torch
torch.set_num_threads(1) # 3. specify multi-threading

with torch.no_grad(): # 4. disable gradients
    sentiment_scores = df['spacy_cln_txt'].apply(sentiment_analyzer)


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

c:\Users\abhis\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\abhis\.cache\huggingface\hub\models--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

CPU times: total: 2min 6s
Wall time: 4min 10s


# 2. NAMED ENTITY RECOGNITION (NER)

Named Entity Recognition (NER) is used to find and label important information (people, places, organizations, dates, etc.) in text
- The default LLM for NER is BERT (encoder-only)

Named Entity Recognition (NER)—also known as entity extraction or entity chunking—is a technique in Natural Language Processing (NLP) that automatically identifies and categorizes key information (entities) within unstructured text.

Think of it as an **AI highlighter**. If you give a computer a sentence like **"Elon Musk founded SpaceX in 2002,"** NER allows the machine to recognize "Elon Musk" as a Person, "SpaceX" as an Organization, and "2002" as a Date.

Common Entity Categories:

NER systems are typically trained to recognize several standard types of entities:
- Person (PER): Names of individuals (e.g., "Albert Einstein").
- Organization (ORG): Companies, agencies, or institutions (e.g., "Google", "NASA").
- Location (LOC/GPE): Geographical areas like cities, countries, or regions (e.g., "Paris", "India").
- Time/Date: Specific dates or timeframes (e.g., "5th May 2025").
- Quantities: Percentages, monetary values, or measurements (e.g., "$100", "50%").

Modern NER systems use three main approaches to "understand" text:
- Dictionary-Based: The system checks text against a pre-existing list (dictionary) of known entities. This is precise but struggles with new or misspelled names.
- Rule-Based: Uses human-written patterns (e.g., "If a word starts with a capital letter and follows 'Mr.', it is likely a Person").
- Machine Learning/Deep Learning: Models are trained on massive datasets to learn the context surrounding words. This allows them to handle ambiguity, such as distinguishing between Amazon the company and the Amazon rainforest.

NER is a foundational step for many advanced AI tasks:
- Search Engines: Helps search engines like Google and Bing provide more relevant results by understanding the entities in your query.
- Customer Support: Automatically categorizes support tickets by product name or location to route them to the right team.
- Content Recommendations: Platforms like Netflix use it to suggest shows based on the actors or genres mentioned in descriptions.
- Healthcare: Extracts patient information, symptoms, and medications from clinical notes for faster analysis.

In [ ]:
text = 'On 12 May 2026 at 10:16 PM in Sadar, \
Uttar Pradesh, Ramu, a senior engineer at Antrhopol, \
was found murdered near the company’s office, with \
    investigators noting a knife wound measuring 12 cm, \
        and evidence suggesting a financial dispute \
            involving ₹125,486.'

from transformers import pipeline

ner_analyzer = pipeline(
    task = "ner",
    model="dbmdz/bert-large-cased-finetuned-conll03-english",
    device=-1,
    aggregation_strategy="SIMPLE"
)


ner_analyzer(text)


config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

c:\Users\abhis\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\abhis\.cache\huggingface\hub\models--dbmdz--bert-large-cased-finetuned-conll03-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.33GB            

model.safetensors: downloading bytes:           |  0.00B            

In [ ]:
# find the named entities in each review
ner_analyzer = pipeline(task = "ner",
    model="dbmdz/bert-large-cased-finetuned-conll03-english",
    device=-1, # run on CPU
    aggregation_strategy='SIMPLE')

# apply to all reviews
df['Named_Entities'] = df['spacy_cln_txt'].apply(lambda text:
    [entity['word'] for entity in ner_analyzer(text)])

# create a unique list of named entities
named_entities = list(set(df.Named_Entities.explode().dropna().tolist()))

# exclude subwords from the list
named_entities_clean = [entity for entity in named_entities if '#' not in entity]
sorted(named_entities_clean)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

#### NAMED ENTITY RECOGNITION (NER)

Mathematical Understanding
TODO

ASSIGNMENT:
1. Read in the children’s books data set
2. Apply NER to the Description column
3. Create a list of all named entities
4. Only include the people (PER)
5. Extra credit: Exclude the authors as well

Zero-Shot Learning (ZSL):

Mathematical Understanding
TODO

**Definition**: A method where a model can classify or understand categories it has never seen during training, by leveraging auxiliary information such as text descriptions, semantic embeddings, or attributes.

**Key Idea**: Instead of training on labeled examples for every possible class, the model learns a **general representation** (often via language models or embeddings) that lets it reason about unseen classes.

Example:
- You train a model on animals like cats and dogs.
- Later, you ask it to classify zebras without zebra training data.
- The model uses semantic knowledge (e.g., “zebra = horse-like animal with stripes”) to make the prediction.

How Does Zero-Shot Learning Work?

- Semantic Space: Classes are described using attributes or natural language (e.g., “fruit that is yellow and curved” → banana).
- Embedding Alignment: Both input data (images, text) and class descriptions are mapped into the same embedding space.
- Inference: The model compares embeddings of unseen inputs with embeddings of class descriptions to assign labels.

TODO

ASSIGNMENT: ZERO-SHOT CLASSIFICATION

1. Apply zero-shot classification to the Description column
2. Find the number of books in each category and check a few to see if the results make sense

Mathematical Understanding:
TODO



#### TEXT SUMMARIZATION

Text summarization is used to make long bodies of text more concise
- The default LLM for text summarization is BART (encoder-decoder)